Observations = Actions. 
Bayesian Game. 
Roles = Types. 

 

Questions:
- How to model the state, what history is in the state? 
- Why do we need each agent to *assumes* there is some ground truth role assigned to them? Currently it is modelled as each agent having preferences of their actions.

Project Game: 
- Two people work together on a project in class
- Each person can either write or code. There is no cost in each action
- Each person has an individual preference of which actions they prefer to do (which indirectly reflects their cost)
- At each timestep, each agent decides what action to do
- After observing the outcome of their collective action, they might change their action in the next round. 
- Over time, depending on the initial preferences, we observe that agents would default to certain actions, which kind of becomes like a role or norm?

# Do TF2 Scrim 6s vs 6s. 

In [25]:
# Import dependencies
from functools import cache
import jax
import jax.numpy as np
from memo import memo
from enum import Enum
import matplotlib.pyplot as plt

In [26]:
from enum import IntEnum

class STATE(IntEnum):
    START = 0
    WW = 1
    WC = 2
    CW = 3
    CC = 4

class ACTIONS(IntEnum):
    WRITE = 0 
    CODE = 1

@jax.jit
def Tr():
    return 1

@jax.jit
def R(a_alice, a_bob):
    # Define reward matrix
    reward_matrix = np.array([
        [2.0, 10.0],
        [10.0, 2.0]])
    
    # Define cost arrays for each agent
    alice_costs = np.array([0.7, 0.3])
    bob_costs = np.array([0.3, 0.7])
    
    return reward_matrix[a_alice, a_bob] - alice_costs[a_alice] - bob_costs[a_bob]

@jax.jit
def gamma():
    return 0.95

def agent_policy(name):
    if name == "alice":
        return np.array([0.7, 0.3])
    elif name == "bob":
        return np.array([0.3, 0.7])

In [27]:
@cache
@memo
def Q[a_alice: ACTIONS, a_bob: ACTIONS](t):
    # Both agents choose their next actions
    alice: knows(a_alice, a_bob)
    alice: chooses(a_alice_ in ACTIONS, wpp=1.0)
    
    bob: knows(a_alice, a_bob)
    bob: chooses(a_bob_ in ACTIONS, wpp=1.0)
    
    # The Q value is the immediate reward plus discounted future value
    return E[
        R(a_alice, a_bob) + (0.0 if t < 0 else gamma() * Q[alice.a_alice_, bob.a_bob_](t - 1))
    ]

# Pre-compile and test
print("Testing Q function:")
Q(0)
q_vals = Q(10)
print("Q values shape:", q_vals.shape)
print(q_vals)

Testing Q function:
Q values shape: (2, 2)
[[41.963985 49.563988]
 [50.363983 41.963985]]


## Interpretation of Results

The Q-values represent the expected long-term reward for each action pair (Alice's action, Bob's action):

```
Q[WRITE, WRITE] = 41.96  Q[WRITE, CODE] = 49.56
Q[CODE, WRITE]  = 50.36  Q[CODE, CODE]  = 41.96
```

**Key Insights:**
1. **Best outcome**: (CODE, WRITE) = 50.36 - Bob codes, Alice writes
2. **Second best**: (WRITE, CODE) = 49.56 - Alice writes, Bob codes  
3. **Worst outcomes**: When both do the same task (WW or CC) = 41.96

This makes sense given the preference costs:
- Alice prefers writing (cost 0.7 for write, 0.3 for code)
- Bob prefers coding (cost 0.3 for code, 0.7 for write)

The optimal coordination is for **each agent to do what they prefer**, leading to role specialization!

---

Now we add a new action: asking for help. 
We also add a state that is a value from 1 to 100. When the state hits 100 which represents progress towards completion? 
This requires us to modify the state to encode some "history"?

In [28]:
## Optimizing Agents - Each agent maximizes their Q-value

# Alice's Q-function: Alice optimizes, Bob chooses uniformly
@cache
@memo
def Q_alice_opt[a_alice: ACTIONS, a_bob: ACTIONS](t):
    alice: knows(a_alice, a_bob)
    
    # Alice models Bob choosing uniformly in the next round
    alice: thinks[bob: chooses(a_bob_ in ACTIONS, wpp=1.0)]
    
    # Alice optimizes her own action to maximize Q
    alice: chooses(a_alice_ in ACTIONS, to_maximize=0.0 if t < 0 else Q_alice_opt[a_alice_, alice.bob.a_bob_](t - 1))
    
    return E[
        R(a_alice, a_bob) + (0.0 if t < 0 else gamma() * Q_alice_opt[alice.a_alice_, alice.bob.a_bob_](t - 1))
    ]

# Bob's Q-function: Bob optimizes, Alice chooses uniformly  
@cache
@memo
def Q_bob_opt[a_alice: ACTIONS, a_bob: ACTIONS](t):
    bob: knows(a_alice, a_bob)
    
    # Bob models Alice choosing uniformly in the next round
    bob: thinks[alice: chooses(a_alice_ in ACTIONS, wpp=1.0)]
    
    # Bob optimizes his own action to maximize Q
    bob: chooses(a_bob_ in ACTIONS, to_maximize=0.0 if t < 0 else Q_bob_opt[bob.alice.a_alice_, a_bob_](t - 1))
    
    return E[
        R(a_alice, a_bob) + (0.0 if t < 0 else gamma() * Q_bob_opt[bob.alice.a_alice_, bob.a_bob_](t - 1))
    ]

# Compute optimal Q values
print("Alice optimizing (Bob plays uniformly):")
q_alice_opt = Q_alice_opt(20)
print("Shape:", q_alice_opt.shape)
print(q_alice_opt)

print("\nBob optimizing (Alice plays uniformly):")
q_bob_opt = Q_bob_opt(20)
print("Shape:", q_bob_opt.shape)
print(q_bob_opt)

# Find optimal actions
print("\n=== Optimal Actions ===")
print("Alice's best response to uniform Bob:")
alice_best_vs_uniform_bob = q_alice_opt.mean(axis=1).argmax()
print(f"  Alice should: {['WRITE', 'CODE'][alice_best_vs_uniform_bob]}")

print("\nBob's best response to uniform Alice:")
bob_best_vs_uniform_alice = q_bob_opt.mean(axis=0).argmax()
print(f"  Bob should: {['WRITE', 'CODE'][bob_best_vs_uniform_alice]}")

Exception: 